# Rate model outputs — SonarQube + MLScent

**A. Code4ML answers** (sections 1-4, SonarQube + MLScent) and **B. refinement final answers** (section 5,
SonarQube only). Anything already on the Hub is skipped, so rerunning never rates a run twice.

Rates the Code4ML answers of every evaluated run with the same SonarQube + MLScent calls used for
`code4ml_annotated.csv` (`experiment/rating/pipeline/rate_annotated.py`, imported from there), then uploads the ratings next to the run on the Hub.

    HF dimsmauls/code4ml-eval-outputs/<run>/code4ml_outputs.jsonl
      -> CSV (id, code_block, ...)
      -> SonarQube (:8000 / :8001) + MLScent (:8101 / :8102), two halves in parallel
      -> <run>/code4ml_ratings.csv on the Hub

- Only rows with `ast_valid = True` are rated (AST validity gate); the other rows stay in the CSV with empty ratings.
- No MLScent notebook context: model answers do not come from a Kaggle notebook, so `context_imports` is empty and
  `code_blocks.csv` is never read.
- Resumable: a run already on the Hub is skipped; an interrupted run resumes from its `part_1.csv` / `part_2.csv`.

**Before running:** start both SonarQube API containers (`sonar_api1`, `sonar_api2`) and both MLScent containers
(`mlscent-1`, `mlscent-2`). The kernel needs `httpx` and `huggingface_hub`, and a Hugging Face login with write access.

## 1. Configuration

In [1]:
import csv
import json
import sys
import threading
from collections import Counter
from pathlib import Path

import httpx
from huggingface_hub import HfApi, hf_hub_download

HERE = Path.cwd().resolve()                                   # experiment/experiment/rating (this notebook)
RATING_PIPELINE = HERE.parents[1] / "rating" / "pipeline"     # experiment/rating/pipeline: rate_annotated.py
assert (RATING_PIPELINE / "rate_annotated.py").exists(), f"rate_annotated.py not found in {RATING_PIPELINE}"
sys.path.insert(0, str(RATING_PIPELINE))
from rate_annotated import SONAR_FIELDS, merge, mlscent_sender, rate_half, sonar_sender, split_halves  # noqa: E402

OUTPUT_REPO = "dimsmauls/code4ml-eval-outputs"
RUNS = ["vanilla", "few-shot", "qwen-14b"] + [f"{m}/seed-{s}" for m in ("lora", "qlora") for s in (42, 43, 44)]
SONAR = ["http://localhost:8000", "http://localhost:8001"]
MLSCENT = ["http://localhost:8101", "http://localhost:8102"]
BATCH_SIZE = 50
WORK_DIR = HERE / "results"
RATINGS_FILE = "code4ml_ratings.csv"

/Users/dimasmaulana/.venvs/mlx/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the services


In [ ]:
api = HfApi()
on_hub = set(api.list_repo_files(OUTPUT_REPO, repo_type="dataset"))
# every other run with Code4ML outputs on the Hub (hyperparameter runs hp-..., prompting-shot runs ...-<k>shot) is
# added as soon as its outputs are there; pilot runs (pilot/...) are skipped
RUNS += sorted(f.rsplit("/", 1)[0] for f in on_hub
               if f.endswith("/code4ml_outputs.jsonl") and not f.startswith("pilot/")
               and f.rsplit("/", 1)[0] not in RUNS)
todo_code4ml = [run for run in RUNS if f"{run}/{RATINGS_FILE}" not in on_hub]
needed = SONAR + (MLSCENT if todo_code4ml else [])   # MLScent is only used for Code4ML runs not rated yet
for url in needed:
    httpx.get(f"{url}/health", timeout=10).raise_for_status()
print("up:", needed, "| Code4ML runs still to rate:", todo_code4ml or "none")

## 3. Steps

In [3]:
def to_csv(run):
    """Download <run>/code4ml_outputs.jsonl and write it as the CSV rate_annotated expects (id, code_block)."""
    path = hf_hub_download(OUTPUT_REPO, f"{run}/code4ml_outputs.jsonl", repo_type="dataset")
    with open(path) as handle:
        rows = [json.loads(line) for line in handle]
    out = WORK_DIR / run.replace("/", "_") / RATINGS_FILE
    out.parent.mkdir(parents=True, exist_ok=True)
    fields = ["id", "graph_vertex_class", "graph_vertex_subclass", "ast_valid", "code_block"]
    with out.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({"id": row["id"], "graph_vertex_class": row["graph_vertex_class"],
                             "graph_vertex_subclass": row["graph_vertex_subclass"],
                             "ast_valid": row["ast_valid"], "code_block": row["code"]})
    return out


def rate(csv_path):
    """Rate the AST-valid rows on both instance pairs in parallel, then merge the ratings into csv_path."""
    with csv_path.open(newline="", encoding="utf-8") as handle:
        rows = [row for row in csv.DictReader(handle) if row["ast_valid"] == "True"]
    parts = [csv_path.parent / "part_1.csv", csv_path.parent / "part_2.csv"]
    errors = []

    def work(half, sonar_url, mlscent_url, out):
        try:  # no kernel_of / imports: MLScent gets no notebook context
            rate_half(half, {}, out, sonar_sender(sonar_url), mlscent_sender(mlscent_url), BATCH_SIZE)
        except BaseException as exc:  # noqa: BLE001 - reported after both threads stop
            errors.append(exc)

    threads = [threading.Thread(target=work, args=args)
               for args in zip(split_halves(rows), SONAR, MLSCENT, parts)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise RuntimeError(f"rating failed (rerun to resume): {errors}")
    merge(csv_path, parts, require_all=False)  # AST-invalid rows keep empty ratings

## 4. Run

In [4]:
for run in RUNS:
    if run not in todo_code4ml:
        print(f"{run}: already on the Hub, skipping")
        continue
    csv_path = to_csv(run)
    rate(csv_path)
    api.upload_file(path_or_fileobj=str(csv_path), path_in_repo=f"{run}/{RATINGS_FILE}", repo_id=OUTPUT_REPO,
                    repo_type="dataset", commit_message=f"rating {run}")
    with csv_path.open(newline="", encoding="utf-8") as handle:
        rated = [row for row in csv.DictReader(handle) if row["ast_valid"] == "True"]
    print(f"{run}: {len(rated)} rated | combined_rating {dict(sorted(Counter(r['combined_rating'] for r in rated).items()))}")

vanilla: already on the Hub, skipping
few-shot: already on the Hub, skipping
qwen-14b: already on the Hub, skipping
lora/seed-42: already on the Hub, skipping
lora/seed-43: already on the Hub, skipping
lora/seed-44: already on the Hub, skipping
qlora/seed-42: already on the Hub, skipping
qlora/seed-43: already on the Hub, skipping
qlora/seed-44: already on the Hub, skipping
part_1.csv: 50/152
part_2.csv: 50/152
part_1.csv: 100/152
part_2.csv: 100/152
part_1.csv: 150/152
part_2.csv: 150/152
part_1.csv: 152/152
part_2.csv: 152/152
hp-lora-cosine/seed-42: 304 rated | combined_rating {'A': 19, 'B': 56, 'N/A': 229}
part_2.csv: 50/144
part_1.csv: 50/144
part_2.csv: 100/144
part_1.csv: 100/144
part_2.csv: 144/144
part_1.csv: 144/144
hp-lora-lr1e-4/seed-42: 288 rated | combined_rating {'A': 15, 'B': 58, 'N/A': 215}
part_1.csv: 50/152
part_2.csv: 50/152
part_1.csv: 100/152
part_2.csv: 100/152
part_2.csv: 150/152
part_1.csv: 150/152
part_2.csv: 152/152
part_1.csv: 152/152
hp-lora-r32/seed-42: 30

## 5. Refinement final answers (SonarQube only)

For every refinement run on the Hub (`refinement-<method>` = execution feedback, `refinement-sonar-<method>` = execution
+ SonarQube feedback), the **final answer per problem** (the passing one, or the last iteration) is rated with the same
local SonarQube service as everything else, so both loops are compared on one measurement path. The SonarQube measures
recorded on the GPU machine during the loop are not used for this comparison. MLScent is not used (DS-1000 code).

    <refinement run>/refinement_results.jsonl -> final answer per problem -> SonarQube (:8000 / :8001)
      -> <refinement run>/refinement_ratings.csv on the Hub

In [5]:
REFINEMENT_RATINGS_FILE = "refinement_ratings.csv"
REFINEMENT_RUNS = sorted(f.rsplit("/", 1)[0] for f in on_hub
                         if f.startswith("refinement") and f.endswith("/refinement_results.jsonl"))


def final_answers(run):
    with open(hf_hub_download(OUTPUT_REPO, f"{run}/refinement_results.jsonl", repo_type="dataset")) as handle:
        rows = [json.loads(line) for line in handle]
    last = {}
    for row in rows:                                   # rows are in iteration order
        last[row["problem_id"]] = row
    return [last[pid] for pid in sorted(last)]


def sonar_measures(items):
    """{id: code} -> {id: measures}, the two SonarQube instances in parallel."""
    ids = list(items)
    halves, results, errors = split_halves(ids), {}, []

    def work(half, url):
        try:
            send = sonar_sender(url)
            for start in range(0, len(half), BATCH_SIZE):
                batch = half[start:start + BATCH_SIZE]
                results.update(send([{"id": str(i), "code": items[i]} for i in batch]))
        except BaseException as exc:  # noqa: BLE001 - reported after both threads stop
            errors.append(exc)

    threads = [threading.Thread(target=work, args=args) for args in zip(halves, SONAR)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise RuntimeError(f"SonarQube rating failed: {errors}")
    return {i: results.get(str(i), {}) for i in ids}


for run in REFINEMENT_RUNS:
    if f"{run}/{REFINEMENT_RATINGS_FILE}" in on_hub:
        print(f"{run}: already on the Hub, skipping")
        continue
    answers = final_answers(run)
    measures = sonar_measures({a["problem_id"]: a["code"] for a in answers})
    out = WORK_DIR / run.replace("/", "_") / REFINEMENT_RATINGS_FILE
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["problem_id", "final_iteration", "passed", *SONAR_FIELDS, "code_block"])
        writer.writeheader()
        for a in answers:
            m = measures[a["problem_id"]]
            writer.writerow({"problem_id": a["problem_id"], "final_iteration": a["iteration"], "passed": a["passed"],
                             **{f: m.get(f) or "" for f in SONAR_FIELDS}, "code_block": a["code"]})
    api.upload_file(path_or_fileobj=str(out), path_in_repo=f"{run}/{REFINEMENT_RATINGS_FILE}", repo_id=OUTPUT_REPO,
                    repo_type="dataset", commit_message=f"rating {run}")
    rated = [m for m in measures.values() if m.get("ncloc")]
    print(f"{run}: {len(answers)} final answers, {len(rated)} analysed by SonarQube | "
          f"maintainability {dict(sorted(Counter(m.get('maintainability_rating') for m in rated).items()))}")
print(f"done: https://huggingface.co/datasets/{OUTPUT_REPO}")

refinement-lora/seed-42: already on the Hub, skipping
refinement-qlora/seed-42: already on the Hub, skipping
refinement-sonar-lora/seed-42: already on the Hub, skipping
refinement-sonar-qlora/seed-42: already on the Hub, skipping
done: https://huggingface.co/datasets/dimsmauls/code4ml-eval-outputs
